In [ ]:
import tonic
import tonic.transforms as T
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

import snntorch as snn
from snntorch import utils
from snntorch import surrogate

from tqdm import tqdm

batch_size = 2
epochs = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
gesture_map = {
    0: "hand_clapping",
    1: "right_hand_wave",
    2: "left_hand_wave",
    3: "right_hand_clockwise",
    4: "right_hand_counter_clockwise",
    5: "left_hand_clockwise",
    6: "left_hand_counter_clockwise",
    7: "forearm_roll_forward",
    8: "forearm_roll_backward",
    9: "drums",
    10: "guitar",
    #11: "random_other_gestures", # -> not implemented here
}

In [ ]:
def combine_polarities(frames):
    # frames are of shape time_frame, polarity, height, width
    # we can concat both polarities (there might be some overlap, but that's ok!)
    # negative polarities are turned into negative numbers
    polarity_zero = -frames[:, 0, :, :]
    polarity_one = frames[:, 1, :, :]
    # Take the value with the largest magnitude
    mask = np.abs(polarity_zero) > np.abs(polarity_one)
    combined = np.where(mask, polarity_zero, polarity_one)
    
    # print(combined.shape)
    return combined


In [ ]:
def plot_frames(frames):
    n_frames = len(frames)
    fig, axes = plt.subplots(2, n_frames, figsize=(n_frames * 3, 6))
    
    for i, frame in enumerate(frames):
        # frame[0] is polarity 0 and frame[1] is polarity 1
        axes[0, i].imshow(frame[1], cmap='gray')
        axes[0, i].set_title("Polarity OFF (0)")
        axes[1, i].imshow(frame[0], cmap='gray')
        axes[1, i].set_title("Polarity ON (1)")
    plt.tight_layout()
    plt.show()

def plot_combined_frames(combined_frames):
    n_frames = len(combined_frames)
    h, w = combined_frames[0].shape
    _, ax = plt.subplots(1, n_frames, figsize=(n_frames * 3, 3))
    for i, combined_frame in enumerate(combined_frames):
        ax[i].imshow(combined_frame, cmap='gray')
        ax[i].set_title(f"Combined Polarity {i}")
        ax[i].axis('off')
    plt.show()

In [ ]:
class basicSNN(nn.Module):
    '''Simple convolutional SNN, meant to operate on two channels of on and off events, with 128 x 128 resolution.'''
    def __init__(self, num_classes=11, num_channels=2, beta=0.9):
        super(basicSNN, self).__init__()
        self.conv1 = nn.Conv2d(num_channels, 32, kernel_size=3, stride=1, padding=1)
        self.lif1 = snn.Leaky(beta=beta)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.lif2 = snn.Leaky(beta=beta)
        self.fc1 = nn.Linear(64 * 32 * 32, num_classes)
        self.lif3 = snn.Leaky(beta=beta)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        B, T, C, H, W = x.shape # batch size, number of time bins, number of channels(polarities), height, width
        m1 = self.lif1.reset_mem()
        m2 = self.lif2.reset_mem()
        m3 = self.lif3.reset_mem()

        out_spikes = []
        for t in range(T):
            xt = x[:, t] # b, c, h, w
            c1 = self.pool(self.conv1(xt))
            s1, m1 = self.lif1(c1, m1)
            c2 = self.pool(self.conv2(s1))
            s2, m2 = self.lif2(c2, m2)
            f  = self.fc1(s2.view(B, -1))
            s3, m3 = self.lif3(f, m3)
            out_spikes.append(s3)

        out_spikes = torch.stack(out_spikes, dim=1) # b, t, num_classes
        return out_spikes

In [ ]:
sensor_size = tonic.datasets.DVSGesture.sensor_size

transform = T.Compose([
    T.ToFrame(sensor_size=sensor_size, time_window=50 * 10 ** 3)
])

train_dataset = tonic.datasets.DVSGesture(save_to="./data", transform=transform, train=True)
test_dataset = tonic.datasets.DVSGesture(save_to="./data", transform=transform, train=False)
# we must pad the frames to the same size

# frames shape is (time_bins, num_polarities, height, width)
# dataloader will return (batch_size, time_bins, num_polarities, height, width)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=tonic.collation.PadTensors(), num_workers=0, pin_memory=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=tonic.collation.PadTensors(), num_workers=0, pin_memory=False)
model = basicSNN(num_classes=len(gesture_map), num_channels=2).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(epochs):
    model.train()
    epoch_train_loss = 0
    epoch_test_loss = 0
    for event_batches, labels in tqdm(train_loader, total=len(train_loader)):
        event_batches = event_batches.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        out_spikes = model(event_batches)
        # print(event_batches.shape)
        # print(out_spikes.shape) # b, t, num_classes
        out_spikes = out_spikes.sum(dim=1) # sum over time
        # print(f"out spikes are {out_spikes} correct labels are {labels}")
        # print(out_spikes.shape)
        # print(labels.shape)
        out_classes = F.softmax(out_spikes, dim=1)
        loss = criterion(out_spikes.view(-1, len(gesture_map)), labels.view(-1))
        loss.backward()
        epoch_train_loss += loss.item()
        optimizer.step()
    
    print(f"Epoch {epoch}, Train Loss: {epoch_train_loss / len(train_loader)}")
    
    # model.eval()
    # # Testing
    # with torch.no_grad():
    #     for event_batches, labels in tqdm(test_loader, total=len(test_loader)):
    #         event_batches = event_batches.to(device)
    #         labels = labels.to(device)
    #         out_spikes = model(event_batches)
    #         out_spikes = out_spikes.mean(dim=1) # average over time
    #         # print(out_spikes.shape)
    #         loss = criterion(out_spikes.view(-1, len(gesture_map)), labels.view(-1))
    #         epoch_test_loss += loss.item()

    # print(f"Epoch {epoch}, Test Loss: {epoch_test_loss / len(test_loader)}")
